In [7]:
import pandas as pd
from pathlib import Path

# Move from notebooks/ to the project root
PROJECT_DIR = Path.cwd().parent

# Locate the dataset folder
DATA_DIR = PROJECT_DIR / "data" / "raw" / "privacy_reviews"

print("Dataset folder exists:", DATA_DIR.exists())
print("Files:", [f.name for f in DATA_DIR.glob("*.csv")])

# Load the original splits
train = pd.read_csv(DATA_DIR / "original_12756.csv")
val = pd.read_csv(DATA_DIR / "val_1595.csv")
test = pd.read_csv(DATA_DIR / "test_1594.csv")

print("\nDataset shapes:")
print("Training:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Dataset folder exists: True
Files: ['final_data_unbalanced_121374.csv', 'original_12756.csv', 'test_1594.csv', 'val_1595.csv']

Dataset shapes:
Training: (12756, 3)
Validation: (1595, 3)
Test: (1594, 3)


In [8]:
datasets = {
    "Training": train,
    "Validation": val,
    "Test": test
}

for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("\nColumns:", df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nLabel distribution:")
    print(df["Label"].value_counts().sort_index())

    print("\nLabel percentages:")
    print(
        (df["Label"].value_counts(normalize=True).sort_index() * 100)
        .round(2)
    )

    print("\nSample reviews:")
    display(df.head(3))


Training

Columns: ['Review', 'Processed', 'Label']

Missing values:
Review       0
Processed    0
Label        0
dtype: int64

Label distribution:
Label
0    2878
1    3366
2    6512
Name: count, dtype: int64

Label percentages:
Label
0    22.56
1    26.39
2    51.05
Name: proportion, dtype: float64

Sample reviews:


,Review,Processed,Label
0,", the most reliable source information even be...",reliable source information even acquire elon ...,2
1,would great instagram allowed users post reels...,would great instagram allow user post reel can...,0
2,were not able load settings right now. try aga...,not able load setting right try later make change,2



Validation

Columns: ['Review', 'Processed', 'Label']

Missing values:
Review       0
Processed    0
Label        0
dtype: int64

Label distribution:
Label
0    360
1    421
2    814
Name: count, dtype: int64

Label percentages:
Label
0    22.57
1    26.39
2    51.03
Name: proportion, dtype: float64

Sample reviews:


,Review,Processed,Label
0,you broke ! good job! you broke ! you are terr...,broke good job break terrible,2
1,the collect way too much personally identifyin...,collect way much personally identify informati...,2
2,the latest update february just rubbish. they ...,late update february rubbish make switch page ...,2



Test

Columns: ['Review', 'Processed', 'Label']

Missing values:
Review       0
Processed    0
Label        0
dtype: int64

Label distribution:
Label
0    359
1    420
2    815
Name: count, dtype: int64

Label percentages:
Label
0    22.52
1    26.35
2    51.13
Name: proportion, dtype: float64

Sample reviews:


,Review,Processed,Label
0,user unable grant permission send messages : t...,user unable grant permission send message issu...,2
1,this platform deleting open and honest convers...,platform delete open honest conversation propa...,2
2,worst worst,bad bad,2


In [9]:
for name, df in datasets.items():
    print(f"\n{'='*45}")
    print(name)
    print("Shape:", df.shape)

    print("\nLabel distribution:")
    print(df["Label"].value_counts().sort_index())

    print("\nLabel percentages:")
    print(
        (df["Label"].value_counts(normalize=True).sort_index() * 100)
        .round(2)
    )

    print("\nSample original reviews:")
    for review in df["Review"].head(3):
        print("-", review)

    print("\nSample processed reviews:")
    for review in df["Processed"].head(3):
        print("-", review)


Training
Shape: (12756, 3)

Label distribution:
Label
0    2878
1    3366
2    6512
Name: count, dtype: int64

Label percentages:
Label
0    22.56
1    26.39
2    51.05
Name: proportion, dtype: float64

Sample original reviews:
- , the most reliable source information even before being acquired elon musk. but nowadays, more diverse accommodating more people interests. keep the great works, !!!
- would great instagram allowed users post reels that can only viewed once, and then disappear. this would give more control over short videos.
- were not able load settings right now. try again later make those changes.

Sample processed reviews:
- reliable source information even acquire elon musk nowadays diverse accommodate people interest keep great work
- would great instagram allow user post reel can view disappear would give control short video
- not able load setting right try later make change

Validation
Shape: (1595, 3)

Label distribution:
Label
0    360
1    421
2    814
Name: coun

In [10]:
# Check duplicate reviews within each split
for name, df in datasets.items():
    print(f"\n{name}")
    print("Duplicate Review rows:", df["Review"].duplicated().sum())
    print("Duplicate Processed rows:", df["Processed"].duplicated().sum())

# Check overlap between splits
train_reviews = set(train["Review"].str.strip().str.lower())
val_reviews = set(val["Review"].str.strip().str.lower())
test_reviews = set(test["Review"].str.strip().str.lower())

print("\nCross-split overlap:")
print("Train - Validation:", len(train_reviews & val_reviews))
print("Train - Test:", len(train_reviews & test_reviews))
print("Validation - Test:", len(val_reviews & test_reviews))


Training
Duplicate Review rows: 0
Duplicate Processed rows: 27

Validation
Duplicate Review rows: 0
Duplicate Processed rows: 0

Test
Duplicate Review rows: 0
Duplicate Processed rows: 0

Cross-split overlap:
Train - Validation: 0
Train - Test: 0
Validation - Test: 0


In [11]:
# Find duplicate processed texts in training
duplicate_rows = train[
    train["Processed"].duplicated(keep=False)
].sort_values("Processed")

print("Total rows involved:", len(duplicate_rows))

display(duplicate_rows[
    ["Review", "Processed", "Label"]
].head(20))

# Check whether duplicate processed texts have conflicting labels
label_counts = (
    duplicate_rows.groupby("Processed")["Label"]
    .nunique()
)

print(
    "\nDuplicate processed texts with different labels:",
    (label_counts > 1).sum()
)

Total rows involved: 48


,Review,Processed,Label
10906,cannot log into account,can not log account,1
10787,cannot log account again..,can not log account,1
3106,cannot log account,can not log account,1
8407,why cannot log into account,can not log account,1
11002,cannot login account,can not login account,1
9247,cannot login into account,can not login account,1
7105,cannot login with account,can not login account,1
7943,cannot login account.....,can not login account,1
5340,deleted,delete,2
6619,delete now.,delete,2



Duplicate processed texts with different labels: 0


In [12]:
duplicate_label_check = (
    train.groupby("Processed")["Label"]
    .nunique()
)

conflicts = duplicate_label_check[duplicate_label_check > 1]

print("Unique duplicated processed texts:",
      train["Processed"].value_counts().gt(1).sum())

print("Texts with conflicting labels:", len(conflicts))

if len(conflicts) > 0:
    print("\nConflicting examples:")
    display(
        train[train["Processed"].isin(conflicts.index)]
        [["Review", "Processed", "Label"]]
    )
else:
    print("All identical processed texts have consistent labels.")

Unique duplicated processed texts: 21
Texts with conflicting labels: 0
All identical processed texts have consistent labels.


In [13]:
# Check label values and their data types
print("Training labels:", sorted(train["Label"].unique()))
print("Label data type:", train["Label"].dtype)

# Check text data types
print("\nReview data type:", train["Review"].dtype)
print("Processed data type:", train["Processed"].dtype)

# Check installed ML libraries
import sklearn
print("\nScikit-learn version:", sklearn.__version__)

Training labels: [np.int64(0), np.int64(1), np.int64(2)]
Label data type: int64

Review data type: str
Processed data type: str

Scikit-learn version: 1.9.1
